# 00 — Data Setup (segmentation + 448 + lesion-grouped split)

**Inputs:** `MyDrive/melanoma/kaggle.json` (your Kaggle API token).
**Outputs (in `MyDrive/melanoma/data/`):**
- `X_all.npy` (N, 448, 448, 3) uint8 — lesion-cropped, hair-removed RGB at 448×448
- `y_all.npy` (N,) int64 — 1 = melanoma, 0 = otherwise
- `ids_all.npy` (N,) — HAM10000 `image_id` strings, same order as X
- `lesion_ids_all.npy` (N,) — HAM10000 `lesion_id` strings (used by the split)
- `seg_fallback_all.npy` (N,) bool — True where Otsu seg failed and a centre
  crop was substituted
- `idx_train.npy`, `idx_val.npy`, `idx_test.npy` — lesion-grouped stratified
  70/15/15 split (no image of a given lesion appears in more than one partition)

Pipeline per image (one-shot, run once on Colab, persisted to Drive):
1. DullRazor hair removal (morphological black-hat + Telea inpaint).
2. Otsu threshold on LAB-L (inverted) + morphological close/open + largest
   connected component. Sanity gates reject suspicious masks; on rejection a
   centred 80%-side fallback crop is used.
3. Crop to lesion bounding box with 15% margin.
4. Resize to 448×448.
5. BGR -> RGB uint8.

Re-runnable: the cell below detects a wrong-shape X_all.npy and rebuilds it.
The first run takes about 15–25 minutes (10,015 images, CPU-bound).

In [ ]:
# --- Colab setup: ensure the project is on sys.path, mount Drive, load config ---
import os, sys, subprocess
from pathlib import Path

# Either the project is already on disk (uploaded zip / mounted Drive) or we
# clone it from GitHub. We never destroy local changes.
REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
    Path("/content/drive/MyDrive/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Mount Drive (silently re-uses an existing mount on re-run)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except (ImportError, ModuleNotFoundError):
    pass

import config
config.ensure_drive_dirs()
print("Project root:", project_root)
print("Drive root  :", config.DRIVE_ROOT)
print("Data dir    :", config.DATA_DIR)
print("Results dir :", config.RESULTS_DIR)

In [ ]:
!pip install --quiet kaggle tqdm opencv-python-headless scikit-image timm 2>&1 | tail -n 1

In [ ]:
# --- Configure Kaggle CLI from kaggle.json on Drive ---
import os, shutil
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(str(config.KAGGLE_JSON_PATH), "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials installed.")

In [ ]:
# --- Download + unzip HAM10000 to /content (fast SSD) ---
import os, subprocess
from pathlib import Path

scratch = Path(config.LOCAL_SCRATCH); scratch.mkdir(parents=True, exist_ok=True)
zip_path = scratch / "ham.zip"

if not zip_path.exists():
    subprocess.run(["kaggle", "datasets", "download",
                    "-d", config.KAGGLE_DATASET, "-p", str(scratch)],
                   check=True)
    # Kaggle drops a zip whose filename matches the dataset slug
    src = next(scratch.glob("*.zip"))
    src.rename(zip_path)
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(scratch)], check=True)

print("Files in scratch:", sorted(p.name for p in scratch.iterdir())[:8], "...")

In [ ]:
# --- Read metadata, build binary labels, expose lesion_id ---
import pandas as pd
meta = pd.read_csv(scratch / "HAM10000_metadata.csv")
meta["y"] = (meta["dx"] == config.POSITIVE_CLASS).astype("int64")
print("Total:", len(meta))
print(meta["dx"].value_counts())
print("Binary balance:", meta["y"].value_counts().to_dict())
n_unique_lesions = meta["lesion_id"].nunique()
print(f"Unique lesions: {n_unique_lesions}  (multi-image lesions: {len(meta) - n_unique_lesions})")

In [ ]:
# --- Index image_id -> filesystem path (images live in two part folders) ---
img_root_candidates = [
    scratch / "ham10000_images_part_1", scratch / "ham10000_images_part_2",
    scratch / "HAM10000_images_part_1", scratch / "HAM10000_images_part_2",
]
id_to_path = {}
for d in img_root_candidates:
    if d.exists():
        for p in d.glob("*.jpg"):
            id_to_path[p.stem] = p
print("Found", len(id_to_path), "image files.")
assert len(id_to_path) >= len(meta), "Some image_ids missing on disk!"

In [ ]:
# --- Build X_all.npy (hair removal + Otsu segmentation + crop + resize) ---
# Re-builds when the on-disk shape doesn't match config.IMG_SIZE.
import cv2, numpy as np
from tqdm import tqdm
from src.preprocessing import preprocess_for_storage

X_path           = config.DATA_DIR / "X_all.npy"
y_path           = config.DATA_DIR / "y_all.npy"
ids_path         = config.DATA_DIR / "ids_all.npy"
lesion_ids_path  = config.DATA_DIR / "lesion_ids_all.npy"
seg_fb_path      = config.DATA_DIR / "seg_fallback_all.npy"

N = len(meta)
expected_shape = (N, config.IMG_SIZE, config.IMG_SIZE, 3)

needs_rebuild = True
if (X_path.exists() and y_path.exists() and ids_path.exists()
        and lesion_ids_path.exists() and seg_fb_path.exists()):
    try:
        X_check = np.load(X_path, mmap_mode="r")
        if X_check.shape == expected_shape:
            needs_rebuild = False
            print(f"Drive arrays already match {expected_shape} — skipping heavy loop.")
    except Exception as exc:
        print(f"Could not inspect existing X_all.npy ({exc}); rebuilding.")

if not needs_rebuild:
    X = np.load(X_path)
    y = np.load(y_path)
    ids = np.load(ids_path, allow_pickle=True)
    lesion_ids = np.load(lesion_ids_path, allow_pickle=True)
    seg_fallback = np.load(seg_fb_path)
else:
    X = np.empty(expected_shape, dtype=np.uint8)
    y = meta["y"].to_numpy()
    ids = meta["image_id"].to_numpy()
    lesion_ids = meta["lesion_id"].to_numpy()
    seg_fallback = np.zeros(N, dtype=bool)
    for i, image_id in enumerate(tqdm(ids, desc="hair-removal + seg + crop + resize")):
        img_bgr = cv2.imread(str(id_to_path[image_id]))
        X[i], used_fb = preprocess_for_storage(
            img_bgr,
            size=config.IMG_SIZE,
            do_hair_removal=True,
            do_segmentation=True,
            seg_margin_frac=config.SEG_MARGIN_FRAC,
            seg_border_frac=config.SEG_BORDER_FRAC,
            seg_min_area_frac=config.SEG_MIN_AREA_FRAC,
            seg_max_area_frac=config.SEG_MAX_AREA_FRAC,
            seg_fallback_frac=config.SEG_FALLBACK_FRAC,
        )
        seg_fallback[i] = used_fb
    np.save(X_path, X)
    np.save(y_path, y)
    np.save(ids_path, ids)
    np.save(lesion_ids_path, lesion_ids)
    np.save(seg_fb_path, seg_fallback)

n_fb = int(seg_fallback.sum())
print(f"X: {X.shape} {X.dtype}   y: {y.shape}   ids: {ids.shape}")
print(f"Otsu segmentation success: {N - n_fb} / {N} "
      f"({100.0 * (N - n_fb) / N:.2f}%) — fallback used in {n_fb} images.")

In [ ]:
# --- Lesion-grouped stratified 70/15/15 split ---
# HAM10000 has multiple images per lesion (~7,470 unique lesions over 10,015
# images). A naive split stratified on `y` only would place different images
# of the same lesion into different partitions and leak information. We split
# by *lesion*, stratified on the lesion's binary label, then map lesions back
# to image indices.
import numpy as np
from sklearn.model_selection import train_test_split

# Per-lesion binary label (all images of a lesion share the same dx in HAM10000).
lesion_y = meta.groupby("lesion_id")["y"].first()
lesion_ids_unique = lesion_y.index.to_numpy()
lesion_labels = lesion_y.to_numpy()

# Stratified hold-out test (lesion-grouped)
trainval_lesions, test_lesions = train_test_split(
    lesion_ids_unique, test_size=config.TEST_FRAC,
    stratify=lesion_labels, random_state=config.SEED)
trainval_y = lesion_y.loc[trainval_lesions].to_numpy()

# Stratified val carve-out from the train+val pool (lesion-grouped)
val_rel = config.VAL_FRAC / (config.TRAIN_FRAC + config.VAL_FRAC)
train_lesions, val_lesions = train_test_split(
    trainval_lesions, test_size=val_rel,
    stratify=trainval_y, random_state=config.SEED)

# Map lesion partitions back to image-row indices
train_set = set(train_lesions); val_set = set(val_lesions); test_set = set(test_lesions)
img_lesions = meta["lesion_id"].to_numpy()
idx_train = np.where(np.isin(img_lesions, list(train_set)))[0]
idx_val   = np.where(np.isin(img_lesions, list(val_set)))[0]
idx_test  = np.where(np.isin(img_lesions, list(test_set)))[0]

# Defensive overlap check (must all be empty)
assert len(np.intersect1d(idx_train, idx_val))  == 0
assert len(np.intersect1d(idx_train, idx_test)) == 0
assert len(np.intersect1d(idx_val,   idx_test)) == 0
assert set(img_lesions[idx_train]).isdisjoint(set(img_lesions[idx_test]))
assert set(img_lesions[idx_train]).isdisjoint(set(img_lesions[idx_val]))
assert set(img_lesions[idx_val]).isdisjoint(set(img_lesions[idx_test]))

np.save(config.DATA_DIR / "idx_train.npy", idx_train)
np.save(config.DATA_DIR / "idx_val.npy",   idx_val)
np.save(config.DATA_DIR / "idx_test.npy",  idx_test)

def dist(label_arr):
    u, c = np.unique(label_arr, return_counts=True)
    return dict(zip(u.tolist(), c.tolist()))

print(f"train: {len(idx_train):>5} images  {dist(y[idx_train])}  "
      f"({len(train_lesions)} lesions)")
print(f"val  : {len(idx_val):>5} images  {dist(y[idx_val])}  "
      f"({len(val_lesions)} lesions)")
print(f"test : {len(idx_test):>5} images  {dist(y[idx_test])}  "
      f"({len(test_lesions)} lesions)")
print("Lesion-grouped split OK — no lesion appears in more than one partition.")

In [ ]:
# --- Final summary ---
import os
def mb(p): return os.path.getsize(p) / 1e6
total = sum(mb(p) for p in config.DATA_DIR.glob("*.npy"))
print(f"Total Drive footprint: {total:.1f} MB")
for p in sorted(config.DATA_DIR.glob('*.npy')):
    print(f"  {p.name:20s} {mb(p):7.1f} MB")
print("\nData setup complete. You can now run notebooks 01-04 in any order.")